# 🧠 Notebook 02 — LSTM Baseline Model
**Sentiment Analysis API Project**

Builds a BiLSTM sentiment model from scratch with GloVe embeddings.
- Load & preprocess data
- Load GloVe 300d embeddings
- Build BiLSTM architecture with attention
- Train with early stopping
- Evaluate and compare to transformer baseline

In [ ]:
import os, re, json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, f1_score
from collections import Counter
from tqdm.notebook import tqdm
from datasets import load_from_disk, load_dataset

plt.rcParams.update({'figure.facecolor':'#0d1117','axes.facecolor':'#161b22',
                     'text.color':'#c9d1d9','axes.labelcolor':'#c9d1d9',
                     'xtick.color':'#8b949e','ytick.color':'#8b949e',
                     'axes.edgecolor':'#30363d','grid.color':'#21262d'})

DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
GLOVE_PATH = '../data/raw/glove.840B.300d.txt'  # Download from Stanford
MAX_VOCAB  = 30000
MAX_LEN    = 128
EMBED_DIM  = 300
HIDDEN_DIM = 256
N_LAYERS   = 2
DROPOUT    = 0.4
BATCH_SIZE = 128
N_EPOCHS   = 15
LR         = 1e-3
N_CLASSES  = 3

print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}GB')

In [ ]:
# ── Data Loading & Preprocessing ──────────────────────────────────
def clean_text(text):
    text = text.lower().strip()
    text = re.sub(r'http\S+', ' ', text)
    text = re.sub(r'[^a-z0-9\s!?,.]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text

try:
    ds = load_from_disk('../data/raw/sst2')
except:
    ds = load_dataset('sst2')

train_texts = [clean_text(x) for x in ds['train']['sentence']]
train_labels = [lbl * 2 for lbl in ds['train']['label']]  # 0→0(neg), 1→2(pos)
val_texts   = [clean_text(x) for x in ds['validation']['sentence']]
val_labels  = [lbl * 2 for lbl in ds['validation']['label']]

print(f'Train: {len(train_texts):,} | Val: {len(val_texts):,}')
print(f'Sample: "{train_texts[0]}" → label {train_labels[0]}')

In [ ]:
# ── Build Vocabulary ───────────────────────────────────────────────
def tokenize(text): return text.split()

all_tokens = [tok for text in train_texts for tok in tokenize(text)]
freq = Counter(all_tokens)
vocab = ['<PAD>', '<UNK>'] + [w for w, c in freq.most_common(MAX_VOCAB - 2)]
word2idx = {w: i for i, w in enumerate(vocab)}

print(f'Vocabulary size: {len(vocab):,}')
print(f'Top 10 tokens: {list(word2idx.keys())[2:12]}')

def encode(text, max_len=MAX_LEN):
    tokens = tokenize(text)[:max_len]
    ids = [word2idx.get(t, 1) for t in tokens]  # 1 = <UNK>
    ids += [0] * (max_len - len(ids))  # pad
    return ids

In [ ]:
# ── Load GloVe Embeddings ──────────────────────────────────────────
def load_glove(path, vocab, embed_dim=300):
    embeddings = np.random.normal(0, 0.1, (len(vocab), embed_dim)).astype(np.float32)
    embeddings[0] = 0  # PAD = zeros
    found = 0
    print(f'Loading GloVe from {path}...')
    if not os.path.exists(path):
        print(f'⚠ GloVe file not found at {path}')
        print('  Download: https://nlp.stanford.edu/projects/glove/')
        print('  Using random embeddings instead.')
        return embeddings
    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
        for line in tqdm(f, desc='Reading GloVe'):
            parts = line.rstrip().split(' ')
            word = parts[0]
            if word in word2idx:
                embeddings[word2idx[word]] = np.array(parts[1:], dtype=np.float32)
                found += 1
    print(f'✔ Loaded {found:,}/{len(vocab):,} embeddings ({found/len(vocab)*100:.1f}%)')
    return embeddings

glove_matrix = load_glove(GLOVE_PATH, vocab)

In [ ]:
# ── PyTorch Dataset ────────────────────────────────────────────────
class SentimentDataset(Dataset):
    def __init__(self, texts, labels):
        self.X = torch.tensor([encode(t) for t in texts], dtype=torch.long)
        self.y = torch.tensor(labels, dtype=torch.long)

    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

train_ds = SentimentDataset(train_texts, train_labels)
val_ds   = SentimentDataset(val_texts,   val_labels)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=256,        shuffle=False, num_workers=4, pin_memory=True)
print(f'Train batches: {len(train_dl)} | Val batches: {len(val_dl)}')

In [ ]:
# ── BiLSTM + Attention Architecture ──────────────────────────────
class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim * 2, 1)

    def forward(self, lstm_out):  # [B, T, H*2]
        scores = self.attn(lstm_out).squeeze(-1)   # [B, T]
        weights = F.softmax(scores, dim=-1)         # [B, T]
        context = (lstm_out * weights.unsqueeze(-1)).sum(dim=1)  # [B, H*2]
        return context, weights

class BiLSTMSentiment(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, n_layers, n_classes, dropout, pretrained_emb=None):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        if pretrained_emb is not None:
            self.embedding.weight.data.copy_(torch.from_numpy(pretrained_emb))
        self.lstm = nn.LSTM(embed_dim, hidden_dim, n_layers, batch_first=True,
                            bidirectional=True, dropout=dropout if n_layers > 1 else 0)
        self.attention = Attention(hidden_dim)
        self.drop = nn.Dropout(dropout)
        self.fc1  = nn.Linear(hidden_dim * 2, hidden_dim)
        self.fc2  = nn.Linear(hidden_dim, n_classes)
        self.bn   = nn.BatchNorm1d(hidden_dim)

    def forward(self, x):
        emb = self.drop(self.embedding(x))       # [B, T, E]
        out, _ = self.lstm(emb)                  # [B, T, H*2]
        ctx, attn_w = self.attention(out)        # [B, H*2]
        h = self.drop(F.relu(self.bn(self.fc1(ctx))))
        return self.fc2(h), attn_w

model = BiLSTMSentiment(len(vocab), EMBED_DIM, HIDDEN_DIM, N_LAYERS, N_CLASSES, DROPOUT, glove_matrix)
model = model.to(DEVICE)
params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model parameters: {params:,}')
print(model)

In [ ]:
# ── Training Loop ─────────────────────────────────────────────────
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)
criterion = nn.CrossEntropyLoss()
scaler    = torch.cuda.amp.GradScaler()

history = {'train_loss': [], 'val_loss': [], 'val_f1': [], 'val_acc': []}
best_f1, patience_ctr, PATIENCE = 0.0, 0, 4

for epoch in range(1, N_EPOCHS + 1):
    # ── Train ──
    model.train()
    total_loss = 0
    for X, y in tqdm(train_dl, desc=f'Epoch {epoch:02d} [TRAIN]', leave=False):
        X, y = X.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            logits, _ = model(X)
            loss = criterion(logits, y)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update()
        total_loss += loss.item()
    avg_train_loss = total_loss / len(train_dl)

    # ── Validate ──
    model.eval()
    all_preds, all_labels, val_loss = [], [], 0
    with torch.no_grad():
        for X, y in val_dl:
            X, y = X.to(DEVICE), y.to(DEVICE)
            logits, _ = model(X)
            val_loss += criterion(logits, y).item()
            all_preds.extend(logits.argmax(1).cpu().numpy())
            all_labels.extend(y.cpu().numpy())

    avg_val_loss = val_loss / len(val_dl)
    f1  = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    acc = (np.array(all_preds) == np.array(all_labels)).mean()
    scheduler.step(avg_val_loss)

    history['train_loss'].append(avg_train_loss)
    history['val_loss'].append(avg_val_loss)
    history['val_f1'].append(f1)
    history['val_acc'].append(acc)

    print(f'Epoch {epoch:02d} | TrainLoss={avg_train_loss:.4f} | ValLoss={avg_val_loss:.4f} | F1={f1:.4f} | Acc={acc:.4f}')

    if f1 > best_f1:
        best_f1 = f1
        patience_ctr = 0
        torch.save(model.state_dict(), '../models/checkpoints/lstm_best.pt')
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE:
            print(f'Early stopping at epoch {epoch}')
            break

print(f'\n✔ Best F1: {best_f1:.4f}')

In [ ]:
# ── Training Curves ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
epochs = range(1, len(history['train_loss']) + 1)

axes[0].plot(epochs, history['train_loss'], color='#00f5ff', lw=2, label='Train')
axes[0].plot(epochs, history['val_loss'],   color='#ff4757', lw=2, label='Val')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs, history['val_f1'],  color='#00ff88', lw=2)
axes[1].set_title('Validation Macro F1'); axes[1].grid(alpha=0.3)

axes[2].plot(epochs, history['val_acc'], color='#b400ff', lw=2)
axes[2].set_title('Validation Accuracy'); axes[2].grid(alpha=0.3)

for ax in axes:
    ax.set_xlabel('Epoch')

plt.suptitle('BiLSTM Training Curves', fontsize=14)
plt.tight_layout()
plt.savefig('../data/processed/lstm_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Final Evaluation ──────────────────────────────────────────────
model.load_state_dict(torch.load('../models/checkpoints/lstm_best.pt'))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for X, y in val_dl:
        X = X.to(DEVICE)
        logits, _ = model(X)
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_labels.extend(y.numpy())

print(classification_report(all_labels, all_preds, target_names=['NEGATIVE','NEUTRAL','POSITIVE']))

# Save metadata
meta = {'model': 'BiLSTM-GloVe', 'vocab_size': len(vocab),
        'embed_dim': EMBED_DIM, 'hidden_dim': HIDDEN_DIM,
        'best_f1': round(best_f1, 4)}
with open('../models/checkpoints/lstm_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)
print('\n✔ LSTM model saved')